# FHR-SAC on Swimmer — tuned family

MuJoCo `Swimmer-v5` — stock SB3 SAC vs **FHRSAC** on the RL-Zoo
`mujoco-defaults` recipe (SB3 SAC defaults + `learning_starts 10000`, 1e6 steps,
net 2×256) with the zoo's **γ 0.9999** override, 5 seeds
[22, 44, 66, 88, 100]. Swimmer is the outlier of the four: forward progress only
pays off over hundreds of steps, so it needs a near-undiscounted horizon (at
γ 0.99 SAC converges to a ≈50-return local optimum). That long effective horizon
is exactly the long-range temporal structure the recurrence penalty acts on.
RL-Zoo reference SAC score is ≈ 345 at 1e6 steps.

The **baseline arm is bit-for-bit stock SB3 SAC** (`fhr_weight 0`, asserted by
`tests/test_sb3_sac_fhr.py`).

**Why a tuned family.** The first pass (`config_sb3_sac.yaml`, 2 seeds, the full
λ × order grid, analysed in `exp1_fhrsac_results.ipynb`) came back
**mechanism-ambiguous**: baseline 343, λ1·r8 344, λ0.1·r8 337, λ1·r2 307, the
frozen-c control 314 and the order-1 smoothing control 311 — every arm within
noise — while λ0.1·r2 split its two seeds 348 / 48. On two seeds that cannot
separate "FHR is neutral on Swimmer" from "FHR is noisy on Swimmer".

**Arms** (`config_sb3_sac_tuned.yaml`) — λ 10 is dropped (it cost 20–30% in the
first pass) and the budget goes on 5 seeds of the four arms that decide the
mechanism question:

| arm | λ | r | c |
|---|---|---|---|
| **exp1** | 1 | 8 | learned — the best arm of the first pass |
| **exp2** | 0.1 | 8 | learned — the rung below it, same order |
| **exp3** | 1 | 2 | learned — the order floor at the same λ |
| **exp4** | 1 | 2 | **frozen** at `[2.0001, −1.0001]`, the exact Bellman pair for γ 0.9999 |
| **exp5** | 1 | 1 | **frozen** at `[1.0001]` — the **pure-smoothing control** |

exp5 is the load-bearing control: its penalty is ≈ Huber(Q_t − Q_{t−1}), plain
temporal smoothing with no Bellman structure at all. If smoothing were the
mechanism it should match exp3/exp4; if the Bellman recurrence is the mechanism
it should not.

The in-training penalised-window probe (`window_rank_every: 5000`) is on for
every arm, the baseline included — section 7.

## 0 · Launch - train whatever the config defines

In [ ]:
import pathlib, sys
SRC_RUNNERS = pathlib.Path.cwd().parent / "src"
if str(SRC_RUNNERS) not in sys.path:
    sys.path.insert(0, str(SRC_RUNNERS))
import run_sb3_seeds as runner
import yaml

CONFIG = "configs/config_sb3_sac_tuned.yaml"
MANIFEST = "cached/sb3_runs_manifest_sac_tuned.json"
# Every arm below - launched and analysed - is exactly what the config's
# experiment.fhr_experiments block currently defines; nothing is hardcoded.
EXPERIMENTS = sorted(int(k) for k in
                     (yaml.safe_load(open(CONFIG))["experiment"]
                      .get("fhr_experiments") or {}))
print("config experiments:", EXPERIMENTS)

LAUNCH = False
FORCE_EXP = False
if LAUNCH:
    manifest = runner.launch_all(config=CONFIG, experiments=EXPERIMENTS,
                                 max_workers=14, force=FORCE_EXP)
    print(sorted(manifest["runs"]))
else:
    print("LAUNCH = False - analysing existing runs only")

## Setup - the figure toolkit

Every figure in this notebook comes from
[`analysis.visualisations.fhr_figures`](../../../src/analysis/visualisations/fhr_figures.py),
so the four MuJoCo notebooks plot the same family the same way and a fix lands
once. Three conventions matter for reading the plots:

* **Colour identifies the arm.** Not lambda - an earlier version coloured by
  lambda, which painted every curve of a single-lambda grid (Ant's tuned family
  is entirely lambda = 0.1) the same blue. Arms take successive colours from the
  Okabe-Ito colour-blind-safe palette; the baseline is always near-black;
  **dashed = frozen-c control**, solid = learned c.
* **The seed band is mean +- 1 s.e.m.**, not the min-max envelope. Switch with
  `ff.BAND = "ci95" | "iqr" | "minmax"` *before* `load_family`.
* **Every `fig_*` call returns one standalone figure**, saved by `F.save(...)`
  to `figures/<family>/<name>.pdf` and `.png` at 300 dpi with Type-42 fonts -
  drop the PDF straight into the paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "analysis").is_dir())
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from analysis.visualisations import fhr_figures as ff

ff.set_pub_style()
ff.BAND = "sem"          # seed band: "sem" | "ci95" | "iqr" | "minmax"

F = ff.load_family(CONFIG, MANIFEST)
F.summary()

# The sample-efficiency ladder. Read off the TRAINING stream (section 1), not
# the greedy-eval curve - see section 3. Automatic 1-2-5 ladder; pass step=/start= to pin it.
THRESHOLDS = F.auto_thresholds()
print("\nthresholds:", THRESHOLDS)

## 1 · Training curves - the episodes as trained

`rewards.csv`: every training episode's return exactly as the stochastic policy
experienced it (SAC's sampled actions, exploration included), against cumulative
env steps, rolling-mean smoothed. One figure per arm against the shared
baseline, so each panel is a clean two-curve comparison you can paste on its
own. Dots mark where the seed-mean curve first crosses each threshold of the
sample-efficiency ladder; the same crossings are tabulated in section 3.

This is the noisy, un-paired counterpart of section 2's greedy-eval curves -
the gap between the two is the exploration cost, and early-terminated episodes
enter at their actual (short) length.

In [ ]:
# One figure per FHR arm vs the baseline, each saved separately.
for a in F.fhr_arms:
    fig = F.fig_training(a.key, thresholds=THRESHOLDS)
    F.save(fig, f"01_training_{a.key}")
    plt.show()

In [ ]:
# ... and the overlay, for the single-figure version of the same story.
fig = F.fig_overlay("train")
F.save(fig, "01_training_all")
plt.show()

## 2 · Learning curves - greedy evaluation

`eval.csv`: the deterministic policy on fixed reset seeds, so the curves are
paired across arms and the training stream is untouched. Again one figure per
arm against the baseline, then the overlay.

In [ ]:
for a in F.fhr_arms:
    fig = F.fig_eval(a.key)
    F.save(fig, f"02_eval_{a.key}")
    plt.show()

In [ ]:
fig = F.fig_overlay("eval")
F.save(fig, "02_eval_all")
plt.show()

## 3 · Sample efficiency - the claim FHR actually makes

Every prior FHR result in this repo is about **onset / sample efficiency**, not
asymptote. The measurement here is deliberately taken on the **training
stream** (section 1) rather than the greedy-eval curve: the training curve is
what the agent's own experience looks like, it is sampled every episode instead
of every 20k steps, and it is the stream a sample-efficiency claim is about.

For each arm and each threshold we take the first env step at which that seed's
rolling-mean training return reaches it, then average over seeds. A threshold
that only some seeds ever reach is marked - averaging the ones that made it
biases the number downwards, so those points are drawn as open markers and left
off the line rather than allowed to bend it.

In [ ]:
_ = F.table_sample_efficiency(THRESHOLDS)

In [ ]:
fig = F.fig_steps_to_threshold(THRESHOLDS)
F.save(fig, "03_steps_to_threshold")
plt.show()

In [ ]:
# The same numbers as a ratio: > 1 means the arm reached that return in fewer
# environment steps than the baseline.
fig = F.fig_speedup(THRESHOLDS)
F.save(fig, "03_speedup")
plt.show()

## 4 · Final performance - one figure per lambda

Final greedy-eval return, one figure per lambda rung so each is a self-contained
arms-vs-baseline comparison. Bar = seed mean, whisker = +- 1 s.e.m., open dots =
the individual seeds (they are the honest picture of spread on 5 seeds), and the
percentage is the change against the baseline of the same figure.

In [ ]:
for lam in F.lambdas:
    fig = F.fig_final(lam)
    F.save(fig, f"04_final_lambda{lam:g}")
    plt.show()

In [ ]:
# The whole learned-c sweep as one lambda x order map (skipped when the family
# has only one lambda or one order - there is no grid to draw).
if len(F.lambdas) > 1 and len(F.orders) > 1:
    fig = F.fig_grid()
    F.save(fig, "04_grid")
    plt.show()

## 5 · FHR + SAC internals

`train_diagnostics.csv` is one row per gradient step - about 1e6 rows and 180 MB
per run, and the previous version of this notebook re-parsed all of it with
`csv.DictReader` for every one of nine panels. `F.prime_diag_cache()` reads each
file **once** with pandas, reduces it to a per-bin median of every column, gives
it an env-step axis and caches the result to `<run>/diag_binned_600.npz` (about
27 kB). The first call costs a few seconds per run; every call after is
instant, and the panels are a few hundred points per curve instead of a million,
so they render and re-render immediately.

The panel that decides whether the lambda ladder was placed correctly is
**rho = lambda·penalty / TD loss** - the fraction of the critic objective the
recurrence term actually owns. fetch_reach's calibrated pipeline targets
rho in [0.5, 2]; read off which rung of the ladder landed in that band.

In [ ]:
F.prime_diag_cache()      # first call: a few seconds per run. Then cached.
fig = F.fig_internals()
F.save(fig, "05_internals")
plt.show()

In [ ]:
F.table_rho()

## 6 · Rollout Hankel rank - the critic trace and the policy itself

Stacked per-rollout Hankels of the min-twin critic trace `Q(s_t, pi(s_t))` and
of each action dimension of `pi(s_t)`, from the **converged** policy. A rank-r
Hankel sequence satisfies an order-r recurrence, so the measured rank of a
converged policy is the smallest order the penalty can enforce without fighting
the solution - this is what says whether the chosen r was well matched.

This is a *rollout* measurement, on greedy on-policy trajectories. Section 7
measures rank where the penalty is actually applied instead.

In [ ]:
fig, table = F.fig_rollout_hankel(runner)
F.save(fig, "06_rollout_hankel")
plt.show()
print(f"{'arm':30s} rank(Q)  ranks(pi dims)")
for label, rq, pr in table:
    print(f"{label:30s} {rq:7d}  {pr}")

## 7 · Penalised-window Hankel rank - the in-training probe

Rank measured **where the penalty is applied**: on sampled replay windows
(anchor + `window_rank_lags` same-episode predecessors, online critics, buffer
actions) rather than on greedy on-policy rollouts. The probe runs for every arm
**including the lambda = 0 baseline**, which samples and measures the same
windows, so its curve is the control: if FHR operates as a rank constraint, its
arms should push the window rank and the penalty-block tail ratio *below* the
baseline on exactly these windows.

The probe is an **in-training** measurement - it cannot be back-filled from
finished runs. It writes `window_hankel.csv` only when `agent.window_rank_every
> 0` was set in the config the runs were launched from; `F.window_probe_status()`
below says whether this family has it.

In [ ]:
_ = F.window_probe_status()

In [ ]:
if F.has_window_probe:
    for a in F.fhr_arms:
        fig = F.fig_window_rank(a.key)
        F.save(fig, f"07_window_rank_{a.key}")
        plt.show()
    fig = F.fig_window_rank_overlay()
    F.save(fig, "07_window_rank_all")
    plt.show()
    print()
    F.table_window_rank()

## 8 · Compute cost

In [ ]:
import os
rows = []
for k, a in F.arms.items():
    for s, d in F.run_dirs(k):
        ck = d / "checkpoints" / "final.pt"
        if ck.exists():
            rows.append((a.plain, s, (os.path.getmtime(ck)
                                      - os.path.getmtime(d / "config.yaml")) / 60))
for label, s, mins in rows:
    print(f"{label:30s} seed {s}: {mins:6.1f} min")
base_name = F.baseline.plain if F.baseline else None
base = [m for l, _, m in rows if l == base_name]
fhr = [m for l, _, m in rows if l != base_name]
if base and fhr:
    print(f"\nbaseline mean {np.mean(base):.1f} min; FHR-arm mean "
          f"{np.mean(fhr):.1f} min (overhead x{np.mean(fhr)/np.mean(base):.2f})")

## 9 · Final policy rollouts — videos

How the final rollout is made: `record_final_videos` reloads each run's
**final checkpoint** (`checkpoints/final.pt` — the policy exactly as it stood
at the end of the training budget; the SB3 zip is self-describing, FHR
coefficients included), rebuilds the same `_make_env` wrapper stack the run
trained and eval'd on with `render_mode="rgb_array"`, and rolls **one greedy
episode** — the deterministic SAC actor (tanh-squashed mean, no sampling),
i.e. the same policy the `eval.csv` curves score — through gymnasium's
`RecordVideo`, which writes `<run_dir>/videos/epfinal-episode-0.mp4`. The
episode resets with the run's own seed, so each video is one representative
rollout per (arm, seed), not the 10-episode eval average — and on Ant an
early **unhealthy termination cuts the video short**, so video length itself
is a health signal. Only arms with completed manifest runs appear; re-run
after a sweep finishes to add the rest.

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")   # headless MuJoCo rendering
from IPython.display import Video, display
from run_sb3_seeds import record_final_videos

for k, a in F.arms.items():
    try:
        vids = record_final_videos(k, config=CONFIG)
    except RuntimeError as e:   # arm not finished in this config's manifest yet
        print(f"{a.plain} ({k}): skipped - {e}")
        continue
    for seed, path in vids:
        print(f"{a.plain} ({k}) - seed {seed}: {path}")
        display(Video(str(path), embed=True, width=420))

## 10 · Findings

*(fill in after the sweep completes)*